# local_character_single

ローカルのdataset/LoRAを使い、Tailscale越しのOllama Qwen2.5-VLで解析・検証しながら、SD1.5 + LoRA + ControlNet + img2img + optional IP-Adapter + inpaint repairで1人キャラの本番1枚生成を行うNotebookです。

Kaggle/Colabのdataset取得や認証は使いません。画像、prompt、dataset、生成結果はローカル側に保持します。

## 使い捨てOllama + Tailscaleコンテナ

サーバー側で実行します。Docker volumeは使いません。コンテナ停止後、Ollamaモデルも消えます。起動中は `OLLAMA_KEEP_ALIVE=0` とAPIリクエスト側の `keep_alive: 0` で、未使用時にモデルを保持し続けない運用にします。

```bash
IMG='ollama/ollama:latest'; \
docker pull "$IMG" && \
CID="$(docker run -d --rm \
  --gpus all \
  --device=/dev/net/tun \
  --cap-add=NET_ADMIN \
  --cap-add=NET_RAW \
  --security-opt=no-new-privileges \
  --tmpfs /tmp:rw,nosuid,nodev \
  --tmpfs /run:rw,nosuid,nodev \
  -e OLLAMA_HOST=0.0.0.0:11434 \
  -e OLLAMA_KEEP_ALIVE=0 \
  --entrypoint /bin/bash \
  "$IMG" \
  -lc '
    set -e

    echo "===== INSTALLING DEPENDENCIES ====="
    apt-get update
    DEBIAN_FRONTEND=noninteractive \
      apt-get install -y --no-install-recommends \
      curl ca-certificates

    echo "===== INSTALLING TAILSCALE ====="
    curl -fsSL https://tailscale.com/install.sh | sh

    mkdir -p /tmp/tailscale

    tailscaled \
      --state=mem: \
      --socket=/tmp/tailscale/tailscaled.sock \
      >/tmp/tailscaled.log 2>&1 &

    sleep 3

    echo
    echo "===== TAILSCALE LOGIN ====="
    tailscale \
      --socket=/tmp/tailscale/tailscaled.sock \
      up \
      --hostname=ollama

    echo
    echo "===== STARTING OLLAMA ====="
    ollama serve >/tmp/ollama.log 2>&1 &
    OLLAMA_PID=$!

    sleep 5

    echo
    echo "===== DOWNLOADING QWEN VLM ====="
    ollama pull qwen2.5vl:7b

    echo
    echo "========================================"
    echo " QWEN VLM SERVER READY"
    echo " Tailscale hostname : ollama"
    echo " Ollama API         : http://ollama:11434"
    echo " Model              : qwen2.5vl:7b"
    echo " Keep alive         : 0"
    echo " Persistent volume  : none"
    echo "========================================"
    echo

    wait "$OLLAMA_PID"
  ')" && \
echo "Docker container: $CID" && \
echo "Waiting for Tailscale authentication..." && \
docker logs -f "$CID"; \
docker wait "$CID" >/dev/null 2>&1 || true; \
echo "===== CLEANUP ====="; \
docker rm -f "$CID" >/dev/null 2>&1 || true; \
docker image rm "$IMG" >/dev/null 2>&1 || true; \
docker builder prune -f >/dev/null 2>&1 || true; \
unset CID IMG; \
echo "===== CLEANUP COMPLETE ====="
```

Notebook側は `OLLAMA_BASE_URL = "http://ollama:11434"` を指定します。

## LoRAの役割別スロット

LoRA学習はこのNotebookでは行いません。必要に応じて事前に作成し、各キャラフォルダ直下に置きます。生成時には以下の役割別LoRAを自動検出します。

```text
identity/base : キャラ全体のidentity
face          : 顔・目・表情
look          : 見た目、髪、色、全体印象
fullbody      : 全身バランス、体型、立ち絵
style         : 画風
outfit        : 衣装
fallback      : 従来の単一キャラLoRA
```

保存先例:

```text
./<dataset>/<character>/<character>-base.safetensors
./<dataset>/<character>/<character>-face.safetensors
./<dataset>/<character>/<character>-look.safetensors
./<dataset>/<character>/<character>-fullbody.safetensors
./<dataset>/<character>/<character>-style.safetensors
./<dataset>/<character>/<character>-outfit.safetensors
./<dataset>/<character>/<character>.safetensors
```

`prompt.md` で明示指定もできます。

```markdown
## lora
base: <character>-base.safetensors
face: <character>-face.safetensors
look: <character>-look.safetensors
fullbody: <character>-fullbody.safetensors
style: <character>-style.safetensors
outfit: <character>-outfit.safetensors

base_weight: 0.70
face_weight: 0.35
look_weight: 0.45
fullbody_weight: 0.35
style_weight: 0.25
outfit_weight: 0.30
```


## 編集する設定
このセルを主に編集します。Kaggle/Colab設定はありません。

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

DATASET_ROOT = Path("./dataset")  # 実データセットrootを指定
LORA_DIR = None  # 通常編集不要。実行時に DATASET_ROOT / CHARACTER へ解決
PROMPT_MD_PATH = Path("./prompt.md")
OUTPUT_ROOT = Path("./ai-image-lab-work/output/local_character_single")

CHARACTER = ""  # 空なら prompt.md の settings.character を使う

OLLAMA_BASE_URL = "http://ollama.tail1cadae.ts.net:11434"
OLLAMA_MODEL = "qwen2.5vl:7b"
OLLAMA_TIMEOUT = 300

MODEL_PRESET = "counterfeit_v30"
DEFAULT_ASPECT = "portrait_3_4"
QUALITY_PRESET = "standard"  # low_vram / standard / high
EXECUTION_MODE = "auto"  # auto / local_cuda / local_mps / cpu_debug

CANDIDATE_COUNT = 6
GEN_STEPS_SEARCH = 24
GEN_STEPS_FINAL = 32
GUIDANCE_SCALE = 6.5

USE_CONTROLNET = True
CONTROLNET_MODE = "lineart"  # lineart / canny
CONTROLNET_SCALE = 0.45
CANNY_LOW_THRESHOLD = 100
CANNY_HIGH_THRESHOLD = 200

USE_IP_ADAPTER = "auto"  # auto / true / false
IP_ADAPTER_SCALE = 0.30
IP_ADAPTER_REPO = "h94/IP-Adapter"
IP_ADAPTER_SUBFOLDER = "models"
IP_ADAPTER_WEIGHT_NAME = "ip-adapter_sd15.bin"

RUN_REPAIR_INPAINT = True
REPAIR_MAX_REGIONS = 1
REPAIR_STRENGTH = 0.42

RUN_UPSCALE = False
UPSCALE_TARGET_LONG_EDGE = 2048

MANUAL_SELECTED_CANDIDATE = ""  # 例: "003.png"。空ならVLMのbest_candidateを使う


## 通常編集不要: プリセット

In [ ]:
ASPECT_PRESETS = {
    "square": (768, 768),
    "portrait_3_4": (768, 1024),
    "landscape_4_3": (1024, 768),
}

ASPECT_PRESETS_LOW_VRAM = {
    "square": (640, 640),
    "portrait_3_4": (672, 896),
    "landscape_4_3": (896, 672),
}

MODEL_PRESETS = {
    "counterfeit_v30": {
        "model_id": "stablediffusionapi/counterfeit-v30",
        "clip_skip": 2,
        "prompt_style": "danbooru",
    },
    "anything_v3": {
        "model_id": "admruul/anything-v3.0",
        "clip_skip": 2,
        "prompt_style": "danbooru",
    },
}

QUALITY_POLICY = {
    "low_vram": {
        "aspect_presets": ASPECT_PRESETS_LOW_VRAM,
        "steps_search": min(GEN_STEPS_SEARCH, 24),
        "steps_final": min(GEN_STEPS_FINAL, 30),
        "candidate_count": min(CANDIDATE_COUNT, 6),
        "allow_ip_adapter": False,
        "repair_regions": min(REPAIR_MAX_REGIONS, 1),
    },
    "standard": {
        "aspect_presets": ASPECT_PRESETS,
        "steps_search": GEN_STEPS_SEARCH,
        "steps_final": GEN_STEPS_FINAL,
        "candidate_count": CANDIDATE_COUNT,
        "allow_ip_adapter": True,
        "repair_regions": REPAIR_MAX_REGIONS,
    },
    "high": {
        "aspect_presets": ASPECT_PRESETS,
        "steps_search": max(GEN_STEPS_SEARCH, 28),
        "steps_final": max(GEN_STEPS_FINAL, 36),
        "candidate_count": max(CANDIDATE_COUNT, 6),
        "allow_ip_adapter": True,
        "repair_regions": max(REPAIR_MAX_REGIONS, 2),
    },
}

IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
MODEL_CFG = MODEL_PRESETS[MODEL_PRESET]
MODEL_ID = MODEL_CFG["model_id"]
CLIP_SKIP = MODEL_CFG["clip_skip"]

## 通常編集不要: 依存関係と環境検出

In [ ]:
import base64
import gc
import io
import json
import math
import os
import re
import shutil
import subprocess
import sys
import textwrap
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import requests
from PIL import Image, ImageFilter, ImageOps


def ensure_package(import_name, install_name=None):
    try:
        __import__(import_name)
        return
    except ImportError:
        pass
    pkg = install_name or import_name
    print(f"installing: {pkg}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)


for import_name, install_name in [
    ("torch", "torch torchvision torchaudio"),
    ("diffusers", "diffusers[torch]"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("safetensors", "safetensors"),
    ("cv2", "opencv-python"),
    ("numpy", "numpy"),
    ("controlnet_aux", "controlnet_aux"),
]:
    try:
        ensure_package(import_name, install_name)
    except Exception as exc:
        print(f"依存関係の確認/導入に失敗: {import_name}: {exc}")

import cv2
import numpy as np
import torch


def detect_backend():
    if EXECUTION_MODE == "cpu_debug":
        return "cpu", "cpu", torch.float32
    if torch.cuda.is_available() and EXECUTION_MODE in ("auto", "local_cuda"):
        return "cuda", "cuda", torch.float16
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() and EXECUTION_MODE in ("auto", "local_mps"):
        return "mps", "mps", torch.float16
    return "cpu", "cpu", torch.float32


BACKEND, DEVICE, TORCH_DTYPE = detect_backend()
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Backend:", BACKEND)
print("Device:", DEVICE)
print("Dtype:", TORCH_DTYPE)
if BACKEND == "cuda":
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} VRAM={props.total_memory / 1024**3:.1f}GB")
elif BACKEND == "mps":
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
    print("MPS fallback:", os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK"))
else:
    print("CPU debug mode: 生成セルは原則スキップしてください。")


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

## 通常編集不要: 出力ディレクトリ

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
SPEC_DIR = RUN_DIR / "specs"
CONTROL_DIR = RUN_DIR / "control"
CANDIDATE_DIR = RUN_DIR / "candidates"
REPAIR_DIR = RUN_DIR / "repair"
LOG_DIR = RUN_DIR / "logs"
for d in [RUN_DIR, SPEC_DIR, CONTROL_DIR, CANDIDATE_DIR, REPAIR_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("RUN_DIR =", RUN_DIR)

## 通常編集不要: prompt.md解析

In [ ]:
def parse_markdown_sections(text):
    sections = {}
    current = None
    buf = []
    for line in text.splitlines():
        m = re.match(r"^##\s+(.+?)\s*$", line)
        if m:
            if current is not None:
                sections[current] = "\n".join(buf).strip()
            current = m.group(1).strip().lower()
            buf = []
        elif current is not None:
            buf.append(line)
    if current is not None:
        sections[current] = "\n".join(buf).strip()
    return sections


def parse_key_values(block):
    data = {}
    if not block:
        return data
    for raw in block.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        data[key.strip().lower()] = value.strip().strip('"').strip("'")
    return data


def strip_md_link(value):
    value = str(value).strip()
    m = re.match(r"^\[.*?\]\((.*?)\)$", value)
    return m.group(1) if m else value


prompt_text = PROMPT_MD_PATH.read_text(encoding="utf-8")
sections = parse_markdown_sections(prompt_text)
settings_cfg = parse_key_values(sections.get("settings", ""))
references_cfg = {k: strip_md_link(v) for k, v in parse_key_values(sections.get("references", "")).items()}
lora_cfg = parse_key_values(sections.get("lora", ""))
generation_cfg = parse_key_values(sections.get("generation", ""))
control_cfg = parse_key_values(sections.get("controlnet", ""))
ip_adapter_cfg = parse_key_values(sections.get("ip_adapter", ""))

PROMPT_POSITIVE_RAW = sections.get("positive", "")
PROMPT_NEGATIVE_RAW = sections.get("negative", "")
PROMPT_RULES_RAW = sections.get("rules", "")

CHARACTER = settings_cfg.get("character", CHARACTER).strip()
if not CHARACTER:
    raise RuntimeError("CHARACTER を設定するか、prompt.md の ## settings に character: <character-folder> を書いてください。")
print("character:", CHARACTER)
print("sections:", sorted(sections.keys()))

(RUN_DIR / "prompt.md").write_text(prompt_text, encoding="utf-8")


## 通常編集不要: 参照画像解決と表示

In [ ]:
def candidate_roots():
    roots = [
        PROMPT_MD_PATH.parent,
        DATASET_ROOT,
        DATASET_ROOT.parent,
        PROJECT_ROOT,
    ]
    unique = []
    for root in roots:
        root = Path(root).expanduser()
        if root not in unique:
            unique.append(root)
    return unique


def resolve_path(value, must_exist=True):
    if not value:
        return None
    value = strip_md_link(value)
    p = Path(value).expanduser()
    if p.is_absolute() and p.exists():
        return p
    for root in candidate_roots():
        q = root / value
        if q.exists():
            return q
    if must_exist:
        raise FileNotFoundError(f"参照パスを解決できません: {value}")
    return p


def list_images(path):
    path = Path(path)
    if path.is_file():
        return [path] if path.suffix.lower() in IMG_EXTS else []
    if not path.exists():
        return []
    return sorted([p for p in path.rglob("*") if p.suffix.lower() in IMG_EXTS])


def first_existing_reference(*names):
    for name in names:
        if name in references_cfg:
            return resolve_path(references_cfg[name])
    return None


identity_image_path = first_existing_reference("identity", "source_image", "portrait")
pose_image_path = first_existing_reference("pose", "control", "control_source", "source_image")
costume_image_path = first_existing_reference("costume", "outfit")
style_path = resolve_path(references_cfg["style"], must_exist=False) if "style" in references_cfg else None
init_image_path = first_existing_reference("init", "source_image", "identity", "portrait")

if identity_image_path is None:
    fallback = DATASET_ROOT / CHARACTER / "portrait"
    imgs = list_images(fallback)
    if imgs:
        identity_image_path = imgs[0]
if init_image_path is None:
    init_image_path = identity_image_path
if pose_image_path is None:
    pose_image_path = init_image_path

style_images = list_images(style_path)[:8] if style_path else []

reference_paths = {
    "identity": identity_image_path,
    "pose": pose_image_path,
    "costume": costume_image_path,
    "style": style_path,
    "init": init_image_path,
}
print(json.dumps({k: str(v) if v else None for k, v in reference_paths.items()}, ensure_ascii=False, indent=2))

preview_items = [("identity", identity_image_path), ("pose", pose_image_path), ("costume", costume_image_path), ("init", init_image_path)]
preview_items = [(k, v) for k, v in preview_items if v and Path(v).exists()]
cols = max(1, len(preview_items))
fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
if cols == 1:
    axes = [axes]
for ax, (label, path) in zip(axes, preview_items):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(label)
    ax.axis("off")
plt.show()

## 通常編集不要: サイズ決定

In [ ]:
def parse_int(value, default=None):
    try:
        return int(str(value).strip())
    except Exception:
        return default


policy = QUALITY_POLICY.get(QUALITY_PRESET, QUALITY_POLICY["standard"])
aspect_presets = policy["aspect_presets"]
aspect = settings_cfg.get("aspect") or generation_cfg.get("aspect") or DEFAULT_ASPECT
width = parse_int(settings_cfg.get("width") or generation_cfg.get("width"))
height = parse_int(settings_cfg.get("height") or generation_cfg.get("height"))
if width is None or height is None:
    width, height = aspect_presets.get(aspect, aspect_presets["portrait_3_4"])

width = int(round(width / 8) * 8)
height = int(round(height / 8) * 8)
GEN_WIDTH, GEN_HEIGHT = width, height
print(f"generation size: {GEN_WIDTH}x{GEN_HEIGHT} aspect={aspect} quality={QUALITY_PRESET}")

## 通常編集不要: Ollama疎通確認

In [ ]:
def ollama_tags():
    url = OLLAMA_BASE_URL.rstrip("/") + "/api/tags"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()


try:
    tags = ollama_tags()
    models = [m.get("name") for m in tags.get("models", [])]
    print("Ollama OK:", OLLAMA_BASE_URL)
    print("models:", models)
    if OLLAMA_MODEL not in models:
        print(f"注意: {OLLAMA_MODEL} が /api/tags に見つかりません。pull済みか確認してください。")
except Exception as exc:
    print("Ollama疎通に失敗。VLM解析/検証セルは失敗します:", exc)

## 通常編集不要: Ollama VLMユーティリティ

In [ ]:
def image_to_base64(path, max_edge=1024):
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_edge, max_edge), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def extract_json_object(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])
    raise ValueError("JSONを抽出できませんでした")


def ollama_chat_json(system_prompt, user_prompt, image_paths=None, temperature=0.1):
    images = [image_to_base64(p) for p in (image_paths or []) if p and Path(p).exists()]
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt, "images": images},
        ],
        "stream": False,
        "keep_alive": 0,
        "options": {"temperature": temperature},
    }
    url = OLLAMA_BASE_URL.rstrip("/") + "/api/chat"
    r = requests.post(url, json=payload, timeout=OLLAMA_TIMEOUT)
    r.raise_for_status()
    content = r.json()["message"]["content"]
    try:
        return extract_json_object(content), content
    except Exception:
        return {"raw": content}, content

## 通常編集不要: Qwen VLM解析

In [ ]:

analysis_system = """
あなたはアニメ・イラストキャラクター画像生成のための画像解析器です。
出力は必ずJSONのみ。説明文やMarkdownは禁止。
参照画像の役割を分離し、維持する要素、変更する要素、禁止する変化を部位単位で整理してください。
Stable Diffusionにそのまま長文を渡す前提ではなく、後段で短いタグへ圧縮するための仕様を作ります。
sd_prompt_hints の positive_short_tags / negative_short_tags は必ず英語の短いDanbooru/SD1.5向けタグで出してください。

さらに、ユーザーが数値を細かく指定しなくても動くように generation_policy と lora_policy を作ってください。
ただし値は保守的にし、画像が参照から崩れすぎない範囲にしてください。
"""

analysis_user = f"""
対象キャラ: {CHARACTER}

prompt.md:
{prompt_text}

次のJSON schemaに合わせて出力してください:
{{
  "character": "...",
  "reference_roles": {{
    "identity": "顔・髪・キャラ性として読む内容",
    "pose": "姿勢・構図として読む内容",
    "costume": "衣装として読む内容",
    "style": "画風として読む内容"
  }},
  "locked": {{
    "identity": [],
    "face": [],
    "eyes": [],
    "hair": [],
    "body": [],
    "colors": [],
    "style": []
  }},
  "editable": [],
  "negative_constraints": [],
  "generation_intent": [],
  "sd_prompt_hints": {{
    "positive_short_tags": [],
    "negative_short_tags": []
  }},
  "lora_policy": {{
    "use_roles": ["base", "face", "look", "fullbody", "style"],
    "disable_roles": ["outfit"],
    "role_weights": {{
      "base": 0.70,
      "face": 0.25,
      "look": 0.35,
      "fullbody": 0.25,
      "style": 0.20,
      "outfit": 0.00
    }},
    "global_scales": [0.85, 1.0],
    "reason": "なぜこの配分にするか"
  }},
  "generation_policy": {{
    "candidate_count": 6,
    "seeds": [42, 43, 44, 45, 46, 47],
    "strengths": [0.38, 0.44, 0.50],
    "controlnet_scales": [0.33, 0.45, 0.57],
    "ip_adapter_scale": 0.35,
    "guidance_scale": 6.5,
    "reason": "なぜこの探索範囲にするか"
  }},
  "verification_checklist": []
}}
"""

analysis_images = [p for p in [identity_image_path, pose_image_path, costume_image_path] if p]
try:
    character_spec, raw_analysis = ollama_chat_json(analysis_system, analysis_user, analysis_images)
except Exception as exc:
    print("VLM解析に失敗。ローカルfallback仕様を使います:", exc)
    raw_analysis = str(exc)
    character_spec = {
        "character": CHARACTER,
        "reference_roles": {},
        "locked": {"identity": [], "face": [], "eyes": [], "hair": [], "body": [], "colors": [], "style": []},
        "editable": [],
        "negative_constraints": [],
        "generation_intent": [],
        "sd_prompt_hints": {"positive_short_tags": [], "negative_short_tags": []},
        "lora_policy": {},
        "generation_policy": {},
        "verification_checklist": [],
    }

(SPEC_DIR / "character_spec.json").write_text(json.dumps(character_spec, ensure_ascii=False, indent=2), encoding="utf-8")
(LOG_DIR / "vlm_analysis_raw.txt").write_text(raw_analysis, encoding="utf-8")
print(json.dumps(character_spec, ensure_ascii=False, indent=2)[:4000])


## 通常編集不要: Prompt Compiler

In [ ]:

QUALITY_TAGS = [
    "masterpiece",
    "best quality",
    "sharp focus",
    "clean lineart",
    "crisp anime illustration",
    "clean cel shading",
    "detailed eyes",
]
NEGATIVE_BASE = [
    "low quality",
    "worst quality",
    "lowres",
    "blurry",
    "soft focus",
    "jpeg artifacts",
    "noise",
    "text",
    "watermark",
    "signature",
    "extra person",
    "multiple girls",
    "bad anatomy",
    "bad hands",
    "extra fingers",
    "missing fingers",
    "deformed fingers",
    "realistic",
    "3d render",
]


def split_tags(text):
    tags = []
    for part in re.split(r",|\n", text or ""):
        part = part.strip(" -\t\r")
        if part:
            tags.append(part)
    return tags


def mostly_ascii(text):
    chars = [c for c in (text or "") if not c.isspace()]
    if not chars:
        return True
    return sum(1 for c in chars if ord(c) < 128) / len(chars) >= 0.80


def dedupe_keep_order(items):
    seen = set()
    out = []
    for item in items:
        item = str(item).strip()
        if not item:
            continue
        key = item.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(item)
    return out


def compact_tags(tags, limit=72):
    tags = dedupe_keep_order(tags)
    return ", ".join(tags[:limit])


spec_hints = character_spec.get("sd_prompt_hints", {}) if isinstance(character_spec, dict) else {}
positive_tags = []
positive_tags.extend(QUALITY_TAGS)
if mostly_ascii(PROMPT_POSITIVE_RAW):
    positive_tags.extend(split_tags(PROMPT_POSITIVE_RAW))
else:
    print("positive本文は日本語/非英語として扱い、SD promptへ直接混ぜずQwenの英語タグを優先します。")
positive_tags.extend(spec_hints.get("positive_short_tags", []) if isinstance(spec_hints, dict) else [])

negative_tags = []
negative_tags.extend(NEGATIVE_BASE)
if mostly_ascii(PROMPT_NEGATIVE_RAW):
    negative_tags.extend(split_tags(PROMPT_NEGATIVE_RAW))
else:
    print("negative本文は日本語/非英語として扱い、SD negativeへ直接混ぜずQwenの英語タグを優先します。")
negative_tags.extend(character_spec.get("negative_constraints", []) if isinstance(character_spec, dict) else [])
negative_tags.extend(spec_hints.get("negative_short_tags", []) if isinstance(spec_hints, dict) else [])

PROMPT_POSITIVE = compact_tags(positive_tags, 72)
PROMPT_NEGATIVE = compact_tags(negative_tags, 90)

compiled_prompt = {
    "positive": PROMPT_POSITIVE,
    "negative": PROMPT_NEGATIVE,
    "positive_tag_count": len(split_tags(PROMPT_POSITIVE)),
    "negative_tag_count": len(split_tags(PROMPT_NEGATIVE)),
}
(SPEC_DIR / "compiled_prompt.json").write_text(json.dumps(compiled_prompt, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(compiled_prompt, ensure_ascii=False, indent=2))


## 通常編集不要: ControlNet画像作成

In [ ]:
def resize_to_generation(image, width, height):
    return ImageOps.contain(image.convert("RGB"), (width, height), Image.LANCZOS, Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS)


def pad_to_size(image, width, height, color=(255, 255, 255)):
    image = image.convert("RGB")
    canvas = Image.new("RGB", (width, height), color)
    x = (width - image.width) // 2
    y = (height - image.height) // 2
    canvas.paste(image, (x, y))
    return canvas


def fit_to_generation(path, width, height):
    img = Image.open(path).convert("RGB")
    img.thumbnail((width, height), Image.LANCZOS)
    return pad_to_size(img, width, height)


def make_canny_control(image, low=100, high=200):
    arr = np.array(image.convert("RGB"))
    edges = cv2.Canny(arr, low, high)
    edges = np.stack([edges, edges, edges], axis=2)
    return Image.fromarray(edges)


def make_lineart_control(image):
    try:
        from controlnet_aux import LineartDetector
        detector = LineartDetector.from_pretrained("lllyasviel/Annotators")
        return detector(image)
    except Exception as exc:
        print(f"lineart作成に失敗。cannyへfallback: {exc}")
        return make_canny_control(image, CANNY_LOW_THRESHOLD, CANNY_HIGH_THRESHOLD)


init_image = fit_to_generation(init_image_path, GEN_WIDTH, GEN_HEIGHT)
control_source = fit_to_generation(pose_image_path or init_image_path, GEN_WIDTH, GEN_HEIGHT)
control_mode = (control_cfg.get("mode") or CONTROLNET_MODE).lower()
if control_mode == "lineart":
    control_image = make_lineart_control(control_source)
else:
    control_image = make_canny_control(control_source, CANNY_LOW_THRESHOLD, CANNY_HIGH_THRESHOLD)

init_image_path_run = CONTROL_DIR / "init_image.png"
control_source_path_run = CONTROL_DIR / "control_source.png"
control_image_path = CONTROL_DIR / "control_image.png"
init_image.save(init_image_path_run)
control_source.save(control_source_path_run)
control_image.save(control_image_path)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, title, img in zip(axes, ["init", "control source", f"control {control_mode}"], [init_image, control_source, control_image]):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
plt.show()
cleanup_memory()

## 通常編集不要: LoRA検出

In [ ]:

def character_lora_dir(character):
    chara_dir = DATASET_ROOT / character
    if not chara_dir.exists():
        matches = [p for p in DATASET_ROOT.rglob(character) if p.is_dir()]
        if matches:
            chara_dir = sorted(matches, key=lambda p: len(p.parts))[0]
    return chara_dir


CHARACTER_LORA_DIR = character_lora_dir(CHARACTER)
LORA_DIR = CHARACTER_LORA_DIR
print("LoRA search dir:", LORA_DIR)


def find_lora_path(name):
    if not name:
        return None
    p = Path(str(name)).expanduser()
    candidates = [p, LORA_DIR / p.name, DATASET_ROOT / CHARACTER / p.name]
    if p.suffix != ".safetensors":
        candidates.extend([q.with_suffix(".safetensors") for q in candidates])
    for q in candidates:
        if q.exists():
            return q
    return None


def find_first_lora(names):
    for name in names:
        path = find_lora_path(name)
        if path:
            return path
    return None


def clamp_float(value, default, lo, hi):
    try:
        value = float(value)
    except Exception:
        value = float(default)
    return max(lo, min(hi, value))


def lora_float(key, default):
    return clamp_float(lora_cfg.get(key, default), default, 0.0, 1.2)


lora_policy = character_spec.get("lora_policy", {}) if isinstance(character_spec, dict) else {}
policy_weights = lora_policy.get("role_weights", {}) if isinstance(lora_policy, dict) else {}
policy_use_roles = {str(x).strip().lower() for x in lora_policy.get("use_roles", [])} if isinstance(lora_policy, dict) else set()
policy_disable_roles = {str(x).strip().lower() for x in lora_policy.get("disable_roles", [])} if isinstance(lora_policy, dict) else set()
prompt_use_roles = {x.strip().lower() for x in lora_cfg.get("use_roles", "").split(",") if x.strip()}
prompt_disable_roles = {x.strip().lower() for x in lora_cfg.get("disable_roles", "").split(",") if x.strip()}
use_roles = prompt_use_roles or policy_use_roles
disable_roles = prompt_disable_roles or policy_disable_roles

LORA_ROLE_PRESETS = [
    {
        "role": "base",
        "aliases": ["base", "identity"],
        "defaults": [f"{CHARACTER}-base.safetensors", f"{CHARACTER}-identity.safetensors"],
        "weight_key": "base_weight",
        "default_weight": 0.70,
        "description": "キャラ全体のidentity",
    },
    {
        "role": "face",
        "aliases": ["face"],
        "defaults": [f"{CHARACTER}-face.safetensors"],
        "weight_key": "face_weight",
        "default_weight": 0.30,
        "description": "顔・目・表情",
    },
    {
        "role": "look",
        "aliases": ["look", "appearance", "hair"],
        "defaults": [f"{CHARACTER}-look.safetensors", f"{CHARACTER}-appearance.safetensors", f"{CHARACTER}-hair.safetensors"],
        "weight_key": "look_weight",
        "default_weight": 0.40,
        "description": "見た目・髪・色・全体印象",
    },
    {
        "role": "fullbody",
        "aliases": ["fullbody", "body", "standing"],
        "defaults": [f"{CHARACTER}-fullbody.safetensors", f"{CHARACTER}-body.safetensors", f"{CHARACTER}-standing.safetensors"],
        "weight_key": "fullbody_weight",
        "default_weight": 0.30,
        "description": "全身バランス・体型・立ち絵",
    },
    {
        "role": "style",
        "aliases": ["style", "artstyle"],
        "defaults": [f"{CHARACTER}-style.safetensors", f"{CHARACTER}-artstyle.safetensors"],
        "weight_key": "style_weight",
        "default_weight": 0.25,
        "description": "画風",
    },
    {
        "role": "outfit",
        "aliases": ["outfit", "costume", "clothes"],
        "defaults": [f"{CHARACTER}-outfit.safetensors", f"{CHARACTER}-costume.safetensors", f"{CHARACTER}-clothes.safetensors"],
        "weight_key": "outfit_weight",
        "default_weight": 0.15,
        "description": "衣装",
    },
]

lora_specs = []
seen_lora_paths = set()
missing_lora_roles = []
disabled_lora_roles = []

for preset in LORA_ROLE_PRESETS:
    role = preset["role"]
    if role in disable_roles or (use_roles and role not in use_roles):
        disabled_lora_roles.append(role)
        continue
    configured = []
    for alias in preset["aliases"]:
        if alias in lora_cfg:
            configured.append(lora_cfg[alias])
    path = find_first_lora(configured + preset["defaults"])
    if path:
        path_key = str(path.resolve())
        if path_key in seen_lora_paths:
            continue
        seen_lora_paths.add(path_key)
        proposed = policy_weights.get(role, preset["default_weight"]) if isinstance(policy_weights, dict) else preset["default_weight"]
        weight = lora_float(preset["weight_key"], proposed)
        lora_specs.append((role, path, weight))
    else:
        missing_lora_roles.append(role)

fallback_lora_path = find_lora_path(lora_cfg.get("fallback", f"{CHARACTER}.safetensors"))
if not lora_specs and fallback_lora_path:
    fallback_weight = policy_weights.get("fallback", 0.80) if isinstance(policy_weights, dict) else 0.80
    lora_specs.append(("character", fallback_lora_path, lora_float("fallback_weight", fallback_weight)))

if not lora_specs:
    raise RuntimeError(f"LoRAが見つかりません: {LORA_DIR} / {CHARACTER}")

print("使用LoRA:")
for name, path, weight in lora_specs:
    desc = next((p["description"] for p in LORA_ROLE_PRESETS if p["role"] == name), "単一キャラLoRA")
    print(f"  {name:8s}: {path} weight={weight}  # {desc}")

if disabled_lora_roles:
    print("無効化LoRA role:", disabled_lora_roles)
if missing_lora_roles:
    print("未検出LoRA role:", missing_lora_roles)
    print("未検出roleはスキップします。必要になったら該当 .safetensors をキャラフォルダへ置いてください。")

(SPEC_DIR / "lora_plan.json").write_text(
    json.dumps(
        {
            "lora_dir": str(LORA_DIR),
            "specs": [{"role": n, "path": str(p), "weight": w} for n, p, w in lora_specs],
            "disabled_roles": disabled_lora_roles,
            "missing_roles": missing_lora_roles,
            "policy_reason": lora_policy.get("reason") if isinstance(lora_policy, dict) else "",
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


## 通常編集不要: Pipelineロード

In [ ]:
if BACKEND == "cpu":
    raise RuntimeError("CPU debug modeです。本番生成にはCUDA/MPS/ROCm相当GPUが必要です。")

from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetImg2ImgPipeline,
    StableDiffusionImg2ImgPipeline,
)


def apply_memory_settings(pipe):
    try:
        pipe.enable_attention_slicing()
    except Exception:
        pass
    try:
        pipe.enable_vae_slicing()
    except Exception:
        pass
    try:
        pipe.enable_vae_tiling()
    except Exception:
        pass
    if BACKEND == "cuda":
        try:
            pipe.enable_xformers_memory_efficient_attention()
            print("xformers enabled")
        except Exception as exc:
            print("xformers unavailable:", exc)
    return pipe


def load_generation_pipeline(use_controlnet=True):
    if use_controlnet:
        controlnet_id = control_cfg.get("model_id")
        if not controlnet_id:
            controlnet_id = "lllyasviel/control_v11p_sd15_lineart" if control_mode == "lineart" else "lllyasviel/sd-controlnet-canny"
        controlnet = ControlNetModel.from_pretrained(controlnet_id, torch_dtype=TORCH_DTYPE)
        pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
            MODEL_ID,
            controlnet=controlnet,
            torch_dtype=TORCH_DTYPE,
            safety_checker=None,
        )
    else:
        controlnet = None
        pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
            MODEL_ID,
            torch_dtype=TORCH_DTYPE,
            safety_checker=None,
        )
    pipe = pipe.to(DEVICE)
    pipe = apply_memory_settings(pipe)
    if hasattr(pipe, "clip_skip"):
        pipe.clip_skip = CLIP_SKIP
    loaded = []
    for adapter_name, lora_path, weight in lora_specs:
        pipe.load_lora_weights(str(lora_path), adapter_name=adapter_name)
        loaded.append(adapter_name)
    pipe.set_adapters(loaded, adapter_weights=[w for _, _, w in lora_specs])
    return pipe, controlnet


pipe, controlnet = load_generation_pipeline(USE_CONTROLNET)

ip_adapter_loaded = False
use_ip = str(USE_IP_ADAPTER).lower()
should_try_ip = use_ip == "true" or (use_ip == "auto" and policy.get("allow_ip_adapter") and BACKEND == "cuda")
if should_try_ip:
    try:
        pipe.load_ip_adapter(
            IP_ADAPTER_REPO,
            subfolder=IP_ADAPTER_SUBFOLDER,
            weight_name=IP_ADAPTER_WEIGHT_NAME,
            image_encoder_folder=None,
        )
        gen_policy = character_spec.get("generation_policy", {}) if isinstance(character_spec, dict) else {}
        proposed_ip_scale = gen_policy.get("ip_adapter_scale", IP_ADAPTER_SCALE) if isinstance(gen_policy, dict) else IP_ADAPTER_SCALE
        pipe.set_ip_adapter_scale(clamp_float(ip_adapter_cfg.get("scale", proposed_ip_scale), proposed_ip_scale, 0.0, 0.8))
        ip_adapter_loaded = True
        print("IP-Adapter loaded")
    except Exception as exc:
        print("IP-Adapterを無効化して続行:", exc)
        ip_adapter_loaded = False

print("pipeline ready", MODEL_ID, "controlnet=", USE_CONTROLNET, "ip_adapter=", ip_adapter_loaded)


## 通常編集不要: 候補6枚生成

In [ ]:

def make_generator(seed):
    if BACKEND == "mps":
        return torch.Generator(device="cpu").manual_seed(seed)
    return torch.Generator(device=DEVICE).manual_seed(seed)


def parse_number_list(value, default, cast=float):
    if value is None or value == "":
        return list(default)
    if isinstance(value, (list, tuple)):
        raw_items = value
    else:
        raw_items = str(value).split(",")
    out = []
    for item in raw_items:
        try:
            out.append(cast(str(item).strip()))
        except Exception:
            pass
    return out or list(default)


def clipped_list(values, default, lo, hi, cast=float):
    out = []
    for value in parse_number_list(values, default, cast):
        try:
            v = cast(value)
            out.append(max(lo, min(hi, v)))
        except Exception:
            pass
    return out or list(default)


def candidate_plan(count):
    gen_policy = character_spec.get("generation_policy", {}) if isinstance(character_spec, dict) else {}
    candidate_cfg = parse_key_values(sections.get("candidate_search", ""))

    default_seeds = [42, 43, 44, 45, 46, 47, 101, 102, 103, 104, 105, 106]
    default_strengths = [0.38, 0.44, 0.50]
    default_control_scales = [CONTROLNET_SCALE, max(0.30, CONTROLNET_SCALE - 0.12), min(0.65, CONTROLNET_SCALE + 0.12)]
    default_lora_scales = [0.85, 1.00]

    seeds = parse_number_list(candidate_cfg.get("seeds", gen_policy.get("seeds")), default_seeds, int)
    strengths = clipped_list(candidate_cfg.get("strengths", gen_policy.get("strengths")), default_strengths, 0.20, 0.70, float)
    control_scales = clipped_list(
        candidate_cfg.get("controlnet_scales", gen_policy.get("controlnet_scales")),
        default_control_scales,
        0.10,
        0.85,
        float,
    )
    lora_policy = character_spec.get("lora_policy", {}) if isinstance(character_spec, dict) else {}
    lora_scales = clipped_list(
        candidate_cfg.get("lora_scales", lora_policy.get("global_scales", gen_policy.get("lora_scales")) if isinstance(lora_policy, dict) else None),
        default_lora_scales,
        0.40,
        1.25,
        float,
    )
    requested_count = parse_int(candidate_cfg.get("limit") or gen_policy.get("candidate_count"), count)
    requested_count = max(1, min(int(requested_count), int(count)))
    guidance = clamp_float(gen_policy.get("guidance_scale", GUIDANCE_SCALE), GUIDANCE_SCALE, 4.0, 9.0)

    print("candidate search policy:")
    print(json.dumps({
        "seeds": seeds[:requested_count],
        "strengths": strengths,
        "controlnet_scales": control_scales,
        "lora_scales": lora_scales,
        "guidance_scale": guidance,
        "generation_policy_reason": gen_policy.get("reason") if isinstance(gen_policy, dict) else "",
    }, ensure_ascii=False, indent=2))

    plan = []
    idx = 0
    while len(plan) < requested_count:
        plan.append({
            "index": len(plan) + 1,
            "seed": seeds[idx % len(seeds)],
            "strength": strengths[idx % len(strengths)],
            "controlnet_scale": control_scales[idx % len(control_scales)],
            "lora_scale": lora_scales[idx % len(lora_scales)],
            "guidance_scale": guidance,
            "steps": policy["steps_search"],
        })
        idx += 1
    return plan


def set_lora_scale(pipe, scale):
    names = [name for name, _, _ in lora_specs]
    base_weights = [weight for _, _, weight in lora_specs]
    pipe.set_adapters(names, adapter_weights=[w * scale for w in base_weights])


def generate_one(plan_item):
    set_lora_scale(pipe, plan_item["lora_scale"])
    generator = make_generator(plan_item["seed"])
    kwargs = dict(
        prompt=PROMPT_POSITIVE,
        negative_prompt=PROMPT_NEGATIVE,
        image=init_image,
        strength=plan_item["strength"],
        guidance_scale=plan_item["guidance_scale"],
        num_inference_steps=plan_item["steps"],
        generator=generator,
    )
    if USE_CONTROLNET:
        kwargs["control_image"] = control_image
        kwargs["controlnet_conditioning_scale"] = plan_item["controlnet_scale"]
    if ip_adapter_loaded and identity_image_path:
        kwargs["ip_adapter_image"] = Image.open(identity_image_path).convert("RGB")
    out = pipe(**kwargs).images[0]
    return out


plans = candidate_plan(policy["candidate_count"])
candidate_records = []
for item in plans:
    idx = item["index"]
    print(f"candidate {idx:03d}: {item}")
    image = generate_one(item)
    image_path = CANDIDATE_DIR / f"{idx:03d}.png"
    meta_path = CANDIDATE_DIR / f"{idx:03d}.json"
    image.save(image_path)
    record = dict(item)
    record["image"] = str(image_path)
    meta_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    candidate_records.append(record)
    del image
    cleanup_memory()

(RUN_DIR / "candidate_plan.json").write_text(json.dumps(candidate_records, ensure_ascii=False, indent=2), encoding="utf-8")
print("generated:", CANDIDATE_DIR)


## 通常編集不要: 候補プレビュー

In [ ]:
candidate_paths = [Path(r["image"]) for r in candidate_records]
cols = min(3, len(candidate_paths))
rows = math.ceil(len(candidate_paths) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 5 * rows))
axes = np.array(axes).reshape(-1)
for ax, path in zip(axes, candidate_paths):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(path.name)
    ax.axis("off")
for ax in axes[len(candidate_paths):]:
    ax.axis("off")
plt.show()

## 通常編集不要: Qwen VLM Verifier

In [ ]:
verify_system = """
あなたはアニメ・イラスト生成結果の厳格な検証器です。
出力は必ずJSONのみ。Markdownは禁止。
参照画像、キャラ仕様、生成指示、候補画像を比較し、最も要件を満たす候補を選んでください。
顔・髪・衣装・単独キャラ・手指・背景・画風・指示遵守を厳しく評価してください。
"""

verify_user = f"""
対象キャラ: {CHARACTER}

compiled_prompt:
{json.dumps(compiled_prompt, ensure_ascii=False)}

character_spec:
{json.dumps(character_spec, ensure_ascii=False)}

候補画像は順番に 001.png, 002.png ... として渡します。

次のJSON schemaで返してください:
{{
  "best_candidate": "003.png",
  "scores": {{
    "001.png": {{"total": 0, "reasons": []}}
  }},
  "failures": [],
  "needs_repair": false,
  "repair_regions": [
    {{"target": "hands", "box": [x1, y1, x2, y2], "prompt": "short repair prompt"}}
  ],
  "repair_prompt": "",
  "recommended_next_action": "accept"
}}
"""

verify_images = [identity_image_path, pose_image_path] + candidate_paths
try:
    verify_report, raw_verify = ollama_chat_json(verify_system, verify_user, verify_images, temperature=0.0)
except Exception as exc:
    print("VLM検証に失敗。001.pngをfallback選択します:", exc)
    raw_verify = str(exc)
    verify_report = {
        "best_candidate": "001.png",
        "scores": {},
        "failures": [str(exc)],
        "needs_repair": False,
        "repair_regions": [],
        "repair_prompt": "",
        "recommended_next_action": "manual_check",
    }

(RUN_DIR / "verify_report.json").write_text(json.dumps(verify_report, ensure_ascii=False, indent=2), encoding="utf-8")
(LOG_DIR / "vlm_verify_raw.txt").write_text(raw_verify, encoding="utf-8")
print(json.dumps(verify_report, ensure_ascii=False, indent=2)[:5000])

## 通常編集不要: Best Candidate選択

In [ ]:
selected_name = MANUAL_SELECTED_CANDIDATE.strip() or verify_report.get("best_candidate", "001.png")
selected_path = CANDIDATE_DIR / selected_name
if not selected_path.exists():
    print(f"指定候補が見つかりません: {selected_path}. 001.pngへfallback")
    selected_path = CANDIDATE_DIR / "001.png"

selected_image = Image.open(selected_path).convert("RGB")
plt.figure(figsize=(5, 7))
plt.imshow(selected_image)
plt.title(f"selected: {selected_path.name}")
plt.axis("off")
plt.show()

## 通常編集不要: 必要時だけInpaint Repair

In [ ]:
from PIL import ImageDraw

final_image = selected_image
repair_records = []

should_repair = bool(RUN_REPAIR_INPAINT and verify_report.get("needs_repair"))
repair_regions = verify_report.get("repair_regions") or []
repair_regions = repair_regions[:policy["repair_regions"]]

if should_repair and repair_regions:
    from diffusers import StableDiffusionInpaintPipeline
    print("loading inpaint pipeline")
    del pipe
    if controlnet is not None:
        del controlnet
    cleanup_memory()
    inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=TORCH_DTYPE,
        safety_checker=None,
    ).to(DEVICE)
    inpaint_pipe = apply_memory_settings(inpaint_pipe)
    for adapter_name, lora_path, weight in lora_specs:
        inpaint_pipe.load_lora_weights(str(lora_path), adapter_name=adapter_name)
    inpaint_pipe.set_adapters([n for n, _, _ in lora_specs], adapter_weights=[w for _, _, w in lora_specs])

    for i, region in enumerate(repair_regions, start=1):
        box = region.get("box")
        if not box or len(box) != 4:
            print("boxなしのrepair regionをスキップ:", region)
            continue
        mask = Image.new("L", final_image.size, 0)
        draw = ImageDraw.Draw(mask)
        x1, y1, x2, y2 = [int(v) for v in box]
        pad = 16
        draw.rectangle([max(0, x1 - pad), max(0, y1 - pad), min(final_image.width, x2 + pad), min(final_image.height, y2 + pad)], fill=255)
        region_prompt = compact_tags(split_tags(PROMPT_POSITIVE) + split_tags(region.get("prompt", "")), 72)
        repaired = inpaint_pipe(
            prompt=region_prompt,
            negative_prompt=PROMPT_NEGATIVE,
            image=final_image,
            mask_image=mask,
            strength=REPAIR_STRENGTH,
            guidance_scale=GUIDANCE_SCALE,
            num_inference_steps=policy["steps_final"],
            generator=make_generator(1000 + i),
        ).images[0]
        mask_path = REPAIR_DIR / f"mask_{i:02d}.png"
        repaired_path = REPAIR_DIR / f"repaired_{i:02d}.png"
        mask.save(mask_path)
        repaired.save(repaired_path)
        final_image = repaired
        repair_records.append({"region": region, "mask": str(mask_path), "image": str(repaired_path)})
        cleanup_memory()

    del inpaint_pipe
    cleanup_memory()
else:
    print("repair skipped")

(RUN_DIR / "repair_records.json").write_text(json.dumps(repair_records, ensure_ascii=False, indent=2), encoding="utf-8")

## 通常編集不要: 最終保存と任意Upscale

In [ ]:
def upscale_pil(image, target_long_edge):
    w, h = image.size
    if max(w, h) >= target_long_edge:
        return image
    scale = target_long_edge / max(w, h)
    new_size = (int(w * scale), int(h * scale))
    up = image.resize(new_size, Image.LANCZOS)
    up = up.filter(ImageFilter.UnsharpMask(radius=1.0, percent=110, threshold=3))
    return up


final_path = RUN_DIR / "final.png"
final_image.save(final_path)

upscale_path = None
if RUN_UPSCALE:
    up = upscale_pil(final_image, UPSCALE_TARGET_LONG_EDGE)
    upscale_path = RUN_DIR / "final_upscaled.png"
    up.save(upscale_path)

metadata = {
    "run_dir": str(RUN_DIR),
    "final_path": str(final_path),
    "upscale_path": str(upscale_path) if upscale_path else None,
    "selected_candidate": str(selected_path),
    "backend": BACKEND,
    "device": DEVICE,
    "model_id": MODEL_ID,
    "lora_specs": [{"name": n, "path": str(p), "weight": w} for n, p, w in lora_specs],
    "size": [GEN_WIDTH, GEN_HEIGHT],
    "controlnet": {"enabled": USE_CONTROLNET, "mode": control_mode, "scale": CONTROLNET_SCALE},
    "ip_adapter_loaded": ip_adapter_loaded,
    "repair_count": len(repair_records),
}
(RUN_DIR / "final_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

plt.figure(figsize=(6, 8))
plt.imshow(final_image)
plt.title("final")
plt.axis("off")
plt.show()

print(json.dumps(metadata, ensure_ascii=False, indent=2))